# Iterative Median Filtering (GPU/CPU) — 3D with optional 2D mode

**What this adds**
- A unified `median_iterative()` that runs on GPU (CuPy) with CPU fallback.
- Works for both **3D volumes** and **2D images** (toggle in preview).
- Ping-pong buffers to avoid per-iteration allocations.
- Slabbed full-dataset pass with overlap that scales with iterations to avoid seams.
- Interactive preview: **Before**, **n iters**, **n+m iters**.

**GPU:** RTX 5080 (16 GB VRAM assumed). If VRAM is tight, reduce slab size.

> Tip: Prefer more *small* iterations over a single huge kernel for better edge preservation.

In [3]:
# Core imports
import os, re, math, pathlib, warnings
from typing import Tuple, Optional, Sequence

import numpy as np
import imageio.v3 as iio
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

# Metrics (optional but handy)
try:
    from skimage.metrics import structural_similarity as ssim
except Exception:
    ssim = None
    warnings.warn("skimage.metrics.ssim not available; SSIM metric will be skipped.")

# GPU (CuPy) with graceful fallback to CPU
_cupy_ok = False
try:
    import cupy as cp
    from cupyx.scipy import ndimage as cnd
    _cupy_ok = True
except Exception as e:
    warnings.warn(f"CuPy not available, falling back to CPU. Reason: {e}")
    cp = None
    cnd = None

# CPU SciPy
from scipy import ndimage as snd

# Widgets for preview UI
try:
    import ipywidgets as widgets
    from IPython.display import display
    _widgets_ok = True
except Exception:
    _widgets_ok = False
    warnings.warn("ipywidgets not available; preview UI will be text-only.")

## Paths & Helper Functions

- **`DATA_DIR`**: base folder for images (2D file or 3D slice stack).
- Helper I/O:
  - `list_files_gray(dir)` → sorted list of slice paths.
  - `read_slice_gray(path)` → grayscale ndarray.
  - `to_float01(x)` / `from_float01(x, dtype)` → convert between `[0,1]` and integer formats.
- Vis/metrics:
  - `show_row(images, titles)` to show 2–4 images row-wise.
  - `quality_metrics_2d/3d` for quick sanity checks.


In [4]:
# ---------- CONFIGURE ROOT DATA DIR ----------
DATA_DIR = "/home/askiran/data/"

# ---------- File helpers ----------
_IMG_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")

def _numeric_key(p: str):
    # natural-ish sort: split digits and words
    parts = re.split(r'(\d+)', os.path.basename(p))
    key = []
    for s in parts:
        key.append(int(s) if s.isdigit() else s.lower())
    return key

def list_files_gray(folder: str):
    files = [str(p) for p in pathlib.Path(folder).glob("*") if p.suffix.lower() in _IMG_EXTS]
    files.sort(key=_numeric_key)
    return files

def read_slice_gray(path: str) -> np.ndarray:
    im = iio.imread(path)
    if im.ndim == 3:  # RGB/RGBA
        im = im[..., :3].mean(axis=-1)
    return im

# ---------- Scaling helpers ----------
def to_float01(x: np.ndarray) -> np.ndarray:
    if np.issubdtype(x.dtype, np.floating):
        # normalize if likely 0..255 or 0..65535
        xmax = float(x.max() if x.size else 1.0)
        if xmax > 1.0:
            return (x.astype(np.float32) / xmax).astype(np.float32)
        return x.astype(np.float32)
    if x.dtype == np.uint8:
        return (x.astype(np.float32) / 255.0).astype(np.float32)
    if x.dtype == np.uint16:
        return (x.astype(np.float32) / 65535.0).astype(np.float32)
    x = x.astype(np.float32)
    xmax = float(x.max() if x.size else 1.0)
    return (x / max(xmax, 1.0)).astype(np.float32)

def from_float01(x: np.ndarray, dtype=np.uint16) -> np.ndarray:
    x = np.clip(x, 0.0, 1.0)
    if dtype == np.uint8:
        return (x * 255.0 + 0.5).astype(np.uint8)
    if dtype == np.uint16:
        return (x * 65535.0 + 0.5).astype(np.uint16)
    return x.astype(dtype)

# ---------- Visualization ----------
def show_row(images, titles=None, figsize=(12,4), cmap="gray", vmin=0, vmax=1):
    n = len(images)
    plt.figure(figsize=figsize)
    for i, im in enumerate(images):
        ax = plt.subplot(1, n, i+1)
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
        if titles:
            ax.set_title(titles[i], fontsize=11)
    plt.tight_layout()
    plt.show()

# ---------- Metrics ----------
def quality_metrics_2d(x: np.ndarray, y: np.ndarray):
    mse = float(np.mean((x - y)**2))
    psnr = 10.0 * math.log10(1.0 / max(mse, 1e-12))
    out = {"mse": mse, "psnr": psnr}
    if ssim is not None:
        try:
            out["ssim"] = float(ssim(x, y, data_range=1.0))
        except Exception:
            pass
    return out

def quality_metrics_3d(vol: np.ndarray, den: np.ndarray):
    mse = float(np.mean((vol - den)**2))
    psnr = 10.0 * math.log10(1.0 / max(mse, 1e-12))
    d = {"mse": mse, "psnr": psnr}
    if ssim is not None:
        try:
            vals = []
            for z in range(vol.shape[0]):
                vals.append(ssim(vol[z], den[z], data_range=1.0))
            d["ssim_mean"] = float(np.mean(vals))
        except Exception:
            pass
    return d

def choose_middle_indices(n: int, k: int):
    if k >= n:
        return list(range(n))
    start = (n - k)//2
    return list(range(start, start + k))

## Iterative median filter (GPU with CPU fallback)

**Algorithm:** apply `median_filter` repeatedly using the same window/footprint.  
**Backend:** CuPy (`cupyx.scipy.ndimage`) when available, otherwise SciPy (`scipy.ndimage`).  
**Buffers:** preallocate two arrays and ping-pong to avoid repeated allocations.

**Parameters**
- `img` *(ndarray)*: 2D or 3D, float32 in `[0,1]` preferred.
- `size` *(int | tuple)*: kernel per axis (odd per axis recommended).  
  - 3D example: `(1,3,3)` to avoid through-slice blur.
  - 2D example: `(3,3)` or `3`.
- `iterations` *(int)*: number of passes (2–4 typical).
- `mode` *(str)*: boundary handling (`reflect`, `nearest`, `mirror`, `wrap`, `constant`).
- `use_gpu` *(bool)*: try GPU first; falls back to CPU on error.
- `preserve_dtype` *(bool)*: cast back to original dtype.

**Notes**
- VRAM use ≈ **2×** the array bytes (input + workspace).
- Prefer increasing **iterations** before using a very large **size**.


In [5]:
def _ensure_tuple_odd(size, ndim):
    if isinstance(size, int):
        size = (size,) * ndim
    size = tuple(max(1, s | 1) for s in size)  # force odd
    assert len(size) == ndim, f"size length {len(size)} != ndim {ndim}"
    return size

def median_iterative(
    img: np.ndarray,
    size: Sequence[int] | int = 3,
    iterations: int = 1,
    *,
    footprint=None,
    mode: str = "reflect",
    use_gpu: bool = True,
    preserve_dtype: bool = True,
):
    """
    Iterative median filter (2D/3D) with GPU first, CPU fallback.
    Accepts NumPy/CuPy arrays; returns same kind (NumPy by default).
    """
    if iterations <= 0:
        return img

    # Detect dimensionality
    if footprint is None:
        size = _ensure_tuple_odd(size, img.ndim)

    # Pick backend
    use_cp = bool(use_gpu and _cupy_ok)
    xp = cp if use_cp else np
    ndi = cnd if use_cp else snd

    # Device transfer if needed
    is_cp_input = (use_cp and isinstance(img, cp.ndarray))
    x = img if (is_cp_input or (not use_cp)) else xp.asarray(img)

    orig_dtype = x.dtype
    if x.dtype != xp.float32:
        x = x.astype(xp.float32, copy=False)

    # Prepare kernel/footprint
    fp = None
    if footprint is not None:
        fp = xp.asarray(footprint)
        ksize = None
    else:
        ksize = size

    # Ping-pong buffers
    cur = x
    out = xp.empty_like(x)

    try:
        for _ in range(int(iterations)):
            ndi.median_filter(cur, size=ksize, footprint=fp, mode=mode, output=out)
            cur, out = out, cur
        y = cur
    except Exception as e:
        # Fallback to CPU if GPU failed (e.g., OOM)
        if use_cp:
            warnings.warn(f"GPU median failed ({e}); running on CPU.")
            return median_iterative(
                img=cp.asnumpy(x) if is_cp_input else np.asarray(img),
                size=size,
                iterations=iterations,
                footprint=None if footprint is None else np.asarray(footprint),
                mode=mode,
                use_gpu=False,
                preserve_dtype=preserve_dtype,
            )
        else:
            raise

    # Convert back dtype if requested
    if preserve_dtype and y.dtype != orig_dtype:
        y = y.astype(orig_dtype, copy=False)

    # Bring back to NumPy if input was NumPy
    if use_cp and not is_cp_input:
        y = cp.asnumpy(y)

    return y

## Config — kernel, iterations, preview and full-run

- `MED_SIZE`: kernel per axis; for 3D try `(1,3,3)`.
- `ITERS_N` + `ITERS_M`: preview shows **n** and **n+m** passes.
- `MED_SLAB`: core slab size for the full run; overlap auto-computed as `ovl_z = (size_z//2) * total_iterations`.

> If you only have 2D images, set `DATA_2D_FILE` to an image path and leave `DATA_3D_DIR=None`.


In [6]:
# -------- Kernel & iterations --------
MED_SIZE = (1, 3, 3)     # for 3D; for 2D, set to (3,3) or just 3 in the preview UI
ITERS_N  = 2             # first pass count for preview
ITERS_M  = 1             # additional passes for preview
TOTAL_ITERS = ITERS_N + ITERS_M

# -------- Dataset selection --------
DATA_3D_DIR  = os.path.join(DATA_DIR, "Peri_1")   # folder of slices for 3D volume
DATA_2D_FILE = None                                # or: "/home/askiran/data/some_image.tif"

# Preview subset
PREVIEW_K_SLICES = 12      # number of middle slices to load for quick preview (3D)
PREVIEW_USE_GPU  = True

# Full run
OUT_MED_PREVIEW = os.path.join(DATA_3D_DIR if DATA_3D_DIR else DATA_DIR, "_median_preview")
OUT_MED_FULL    = os.path.join(DATA_3D_DIR if DATA_3D_DIR else DATA_DIR, "_median_full")
os.makedirs(OUT_MED_PREVIEW, exist_ok=True)
os.makedirs(OUT_MED_FULL, exist_ok=True)

# Slab parameters
MED_SLAB = 50  # core Z-slices per slab for full run (adjust if VRAM is tight)


## Median Preview (interactive) — with selective saving

Choose:
- **Mode**: 3D (applies `(z,y,x)` kernel on a small subvolume) or 2D (applies `(y,x)` kernel on the chosen image).
- **Iterations**: `n` and additional `m`; we display **Before**, **n**, **n+m**.
- **Save selected**: tick which views to save; choose dtype/format and a filename prefix.

**Saves to:** `OUT_MED_PREVIEW`  
**Filenames:** `<prefix>_<srcname>_<view>.tif` (or `.png`), where view ∈ {`before`, `n{n}`, `n{n}_m{m}`}.


In [10]:
def _load_preview_volume_3d(folder: str, k: int) -> np.ndarray:
    files = list_files_gray(folder)
    assert len(files) > 0, f"No image files in {folder}"
    idxs = choose_middle_indices(len(files), k)
    vol = np.stack([to_float01(read_slice_gray(files[i])) for i in idxs], axis=0).astype(np.float32)
    return vol, files, idxs

def _center_slice_index(idxs):
    return len(idxs)//2

# Build preview data
_preview_vol = None
_preview_files = None
_preview_idxs = None
if DATA_3D_DIR and os.path.isdir(DATA_3D_DIR):
    _preview_vol, _preview_files, _preview_idxs = _load_preview_volume_3d(DATA_3D_DIR, PREVIEW_K_SLICES)

# 2D image if specified
_preview_2d = None
if DATA_2D_FILE and os.path.isfile(DATA_2D_FILE):
    _preview_2d = to_float01(read_slice_gray(DATA_2D_FILE)).astype(np.float32)

def _apply_2d(img2d: np.ndarray, ksize_2d, iters: int, use_gpu=True):
    return median_iterative(img2d, size=ksize_2d, iterations=iters, use_gpu=use_gpu)

def _apply_3d_center(vol3d: np.ndarray, ksize_3d, iters: int, z_center: int, use_gpu=True):
    # Determine z half-window and extend a small local slab
    hz = (ksize_3d[0] // 2)
    pad = iters * hz  # iterative dependency depth
    z0 = max(0, z_center - (pad + 1))
    z1 = min(vol3d.shape[0], z_center + (pad + 1) + 1)
    sub = vol3d[z0:z1]
    den = median_iterative(sub, size=ksize_3d, iterations=iters, use_gpu=use_gpu)
    # Map back to center slice index in 'den'
    zi = min(z_center - z0, den.shape[0]-1)
    return den[zi]

def _pick_dtype(src_path: str, choice: str):
    src = read_slice_gray(src_path)
    if choice == "match_source":
        return src.dtype
    if choice == "uint8":   return np.uint8
    if choice == "uint16":  return np.uint16
    if choice == "float32": return np.float32
    return src.dtype

def _safe_write(path: str, arr: np.ndarray):
    # imageio.v3 handles tif/png; ensure parent exists
    os.makedirs(os.path.dirname(path), exist_ok=True)
    iio.imwrite(path, arr)

def median_preview_widget():
    if not _widgets_ok:
        print("ipywidgets not available. Install ipywidgets for the interactive preview.")
        return

    # ---- Controls ----
    mode = widgets.ToggleButtons(options=["3D", "2D"], value="3D", description="Mode:")
    n_box = widgets.IntSlider(value=ITERS_N, min=1, max=12, step=1, description="n iters")
    m_box = widgets.IntSlider(value=ITERS_M, min=0, max=12, step=1, description="m extra")
    kz_box = widgets.IntSlider(value=MED_SIZE[0] if isinstance(MED_SIZE, tuple) else 1, min=1, max=9, step=2, description="kz (odd)")
    ky_box = widgets.IntSlider(value=MED_SIZE[1] if isinstance(MED_SIZE, tuple) else 3, min=1, max=21, step=2, description="ky (odd)")
    kx_box = widgets.IntSlider(value=MED_SIZE[2] if isinstance(MED_SIZE, tuple) else 3, min=1, max=21, step=2, description="kx (odd)")
    use_gpu = widgets.Checkbox(value=PREVIEW_USE_GPU, description="Use GPU")

    if _preview_vol is not None:
        z_center_default = _center_slice_index(_preview_idxs)
        z_sel = widgets.IntSlider(value=z_center_default, min=0, max=len(_preview_idxs)-1, step=1, description="z (preview)")
    else:
        z_sel = widgets.IntSlider(value=0, min=0, max=0, step=1, description="z (preview)")

    # ---- Save controls ----
    save_before = widgets.Checkbox(value=False, description="Save Before")
    save_n      = widgets.Checkbox(value=True,  description="Save n iters")
    save_nm     = widgets.Checkbox(value=False, description="Save n+m iters")
    prefix      = widgets.Text(value="prev", description="Prefix:")
    fmt_dd      = widgets.Dropdown(options=["tif", "png"], value="tif", description="Format:")
    dtype_dd    = widgets.Dropdown(
        options=[("Match source","match_source"), ("uint8","uint8"), ("uint16","uint16"), ("float32","float32")],
        value="match_source", description="Save dtype:"
    )
    save_btn    = widgets.Button(description="Save selected", button_style="success", icon="save")
    status      = widgets.HTML(value="")

    out = widgets.Output()

    # Store the last shown images so the save button can access them
    last = {
        "mode": None,
        "raw": None,
        "den_n": None,
        "den_nm": None,
        "z_idx_in_preview": None,   # index within _preview_idxs
        "abs_z": None,              # absolute z (file index)
        "src_path": None,           # for 3D: file path of the shown slice; for 2D: DATA_2D_FILE
        "n": None,
        "m": None,
        "ksize": None
    }

    def _update(*args):
        with out:
            out.clear_output(wait=True)

            n = max(1, int(n_box.value))
            m = max(0, int(m_box.value))
            ksz = (int(kz_box.value)|1, int(ky_box.value)|1, int(kx_box.value)|1)

            # enable/disable nm checkbox depending on m
            save_nm.disabled = (m == 0)

            if mode.value == "3D":
                if _preview_vol is None:
                    print("No 3D preview volume found (set DATA_3D_DIR).")
                    return
                zc = int(z_sel.value)
                raw_slice = _preview_vol[zc]
                den_n  = _apply_3d_center(_preview_vol, ksz, n,  zc, use_gpu=use_gpu.value)
                den_nm = _apply_3d_center(_preview_vol, ksz, n+m, zc, use_gpu=use_gpu.value)

                titles = [f"Before (z={_preview_idxs[zc]})",
                          f"{n} iters",
                          f"{n+m} iters"]
                show_row([raw_slice, den_n, den_nm], titles)

                # quick metrics vs raw
                qm_n  = quality_metrics_2d(raw_slice, den_n)
                qm_nm = quality_metrics_2d(raw_slice, den_nm)
                print("Metrics vs raw:", {"n": qm_n, "n+m": qm_nm})

                # record save context
                last.update({
                    "mode": "3D",
                    "raw": raw_slice, "den_n": den_n, "den_nm": den_nm,
                    "z_idx_in_preview": zc,
                    "abs_z": _preview_idxs[zc],
                    "src_path": _preview_files[_preview_idxs[zc]],
                    "n": n, "m": m, "ksize": ksz
                })

            else:  # 2D
                if _preview_2d is None:
                    print("No 2D image set (set DATA_2D_FILE).")
                    return
                k2d = (ksz[1], ksz[2])  # (ky,kx)
                den_n  = _apply_2d(_preview_2d, k2d, n, use_gpu=use_gpu.value)
                den_nm = _apply_2d(_preview_2d, k2d, n+m, use_gpu=use_gpu.value)

                titles = ["Before", f"{n} iters", f"{n+m} iters"]
                show_row([_preview_2d, den_n, den_nm], titles)

                # quick metrics vs raw
                qm_n  = quality_metrics_2d(_preview_2d, den_n)
                qm_nm = quality_metrics_2d(_preview_2d, den_nm)
                print("Metrics vs raw:", {"n": qm_n, "n+m": qm_nm})

                last.update({
                    "mode": "2D",
                    "raw": _preview_2d, "den_n": den_n, "den_nm": den_nm,
                    "z_idx_in_preview": None,
                    "abs_z": None,
                    "src_path": DATA_2D_FILE if DATA_2D_FILE else None,
                    "n": n, "m": m, "ksize": k2d  # note: (ky,kx) for 2D
                })

    def _on_save_clicked(b):
        status.value = ""
        if last["raw"] is None:
            status.value = "<span style='color:#c00'>Nothing to save yet — change a control to render first.</span>"
            return

        fmt = fmt_dd.value
        dtype_choice = dtype_dd.value
        src_path = last["src_path"]
        if src_path is None:
            status.value = "<span style='color:#c00'>No source file path available.</span>"
            return

        # Determine dtype
        dtype = _pick_dtype(src_path, dtype_choice)
        src_name = os.path.splitext(os.path.basename(src_path))[0]
        pref = prefix.value.strip() or "prev"

        to_save = []
        if save_before.value:
            to_save.append(("before", last["raw"]))
        if save_n.value:
            to_save.append((f"n{last['n']}", last["den_n"]))
        if (not save_nm.disabled) and save_nm.value:
            to_save.append((f"n{last['n']}_m{last['m']}", last["den_nm"]))

        if not to_save:
            status.value = "<span style='color:#c00'>Select at least one image to save.</span>"
            return

        saved = []
        for tag, img in to_save:
            arr = from_float01(img, dtype=dtype) if np.issubdtype(dtype, np.integer) else img.astype(np.float32, copy=False)

            # PNG doesn't support float32 reliably across viewers; coerce to uint16
            if fmt == "png" and np.issubdtype(arr.dtype, np.floating):
                arr = from_float01(img, dtype=np.uint16)

            if last["mode"] == "3D" and last["abs_z"] is not None:
                fn = f"{pref}_{src_name}_{tag}_z{last['abs_z']:04d}.{fmt}"
            else:
                fn = f"{pref}_{src_name}_{tag}.{fmt}"

            out_path = os.path.join(OUT_MED_PREVIEW, fn)
            _safe_write(out_path, arr)
            saved.append(out_path)

        status.value = "<span style='color:#070'>Saved:</span><br>" + "<br>".join(saved)

    # Wire up
    for w in [mode, n_box, m_box, kz_box, ky_box, kx_box, z_sel, use_gpu]:
        w.observe(_update, names="value")
    save_btn.on_click(_on_save_clicked)

    # Initial render
    _update()

    # Layout
    controls = widgets.HBox([mode, n_box, m_box, use_gpu])
    kernels  = widgets.HBox([kz_box, ky_box, kx_box, z_sel])
    savebar1 = widgets.HBox([save_before, save_n, save_nm])
    savebar2 = widgets.HBox([prefix, fmt_dd, dtype_dd, save_btn])
    ui = widgets.VBox([controls, kernels, widgets.HTML("<b>Preview</b>"), widgets.Output(),])
    display(widgets.VBox([controls, kernels, widgets.HTML("<b>Preview</b>"), out,
                          widgets.HTML("<b>Save selected</b>"),
                          savebar1, savebar2, status]))

median_preview_widget()

## Full 3D median — slabbed processing

- Processes the full stack in **Z-slabs** of `MED_SLAB`.
- Overlap: `ovl_z = (MED_SIZE[0]//2) * TOTAL_ITERS` to avoid seam artifacts after multiple passes.
- Saves output slices to `OUT_MED_FULL` keeping the original bit depth.

> If you only need 2D processing on single images, skip this cell.


In [9]:
def run_full_3d_median(
    folder: str,
    size_zyx: Tuple[int,int,int],
    iterations: int,
    slab: int,
    out_dir: str,
    use_gpu: bool = True,
):
    files = list_files_gray(folder)
    Z = len(files)
    assert Z > 0, f"No images in {folder}"

    # half window per axis
    hz, hy, hx = (s//2 for s in _ensure_tuple_odd(size_zyx, 3))
    ovl_z = hz * max(1, int(iterations))

    print(f"Total slices: {Z} | slab={slab} | overlap(z)={ovl_z} | iters={iterations} | kernel={size_zyx}")

    for z0 in tqdm(range(0, Z, slab), desc="Median3D full"):
        z1 = min(z0 + slab, Z)
        zs = max(0, z0 - ovl_z)
        ze = min(Z, z1 + ovl_z)

        # load extended slab, to float [0,1]
        ext = np.stack([to_float01(read_slice_gray(p)) for p in files[zs:ze]], axis=0).astype(np.float32)

        # filter extended slab
        den_ext = median_iterative(ext, size=size_zyx, iterations=iterations, use_gpu=use_gpu)

        # trim back to core region
        core = den_ext[(z0 - zs):(z1 - zs)]

        # save out with original dtype per-slice
        for i, z in enumerate(range(z0, z1)):
            src = read_slice_gray(files[z])
            dtype = src.dtype
            out_u = from_float01(core[i], dtype=dtype)
            iio.imwrite(os.path.join(out_dir, f"slice_{z:04d}.tif"), out_u)

    print("Saved to:", out_dir)

# Run the full 3D pass (uses TOTAL_ITERS = ITERS_N + ITERS_M)
if DATA_3D_DIR and os.path.isdir(DATA_3D_DIR):
    run_full_3d_median(
        folder=DATA_3D_DIR,
        size_zyx=_ensure_tuple_odd(MED_SIZE, 3),
        iterations=TOTAL_ITERS,
        slab=MED_SLAB,
        out_dir=OUT_MED_FULL,
        use_gpu=True
    )
else:
    print("DATA_3D_DIR not set or not a directory; skipping full 3D run.")


Total slices: 2 | slab=50 | overlap(z)=0 | iters=3 | kernel=(1, 3, 3)


Median3D full:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /home/askiran/data/Peri_1/_median_full


## Troubleshooting & Tuning

- **Edges get rounded:** keep `size_z=1` (i.e., `(1,3,3)`) and increase iterations to 3–4 before enlarging `(ky,kx)`.
- **Seam artifacts between slabs:** increase overlap `ovl_z = (size_z//2) * iterations` by +1–2.
- **GPU OOM:** reduce `MED_SLAB` by 25–50%, or uncheck “Use GPU” in preview for CPU runs.
- **Performance:** VRAM consumption ≈ `2×(extended slab bytes)`. For a `1024×1024×100 float32` slab, that’s ~0.4 GB × 2 = 0.8 GB.
- **2D only:** set `DATA_2D_FILE` and use the preview in “2D” mode; skip the full 3D cell.
